# Environment Setup

In [1]:
!pip -q install datasets==3.6.0
!pip -q install -U huggingface_hub hf_transfer
!pip -q install -U duckdb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.5 MB/s eta 0:00:00


In [2]:
# Importing all the dependencies
from datasets import load_dataset
import pandas as pd
import numpy as np
from typing import Tuple, Optional, Union, Dict, List
import os, time, duckdb
from huggingface_hub import login, HfApi, list_repo_files
import gdown
from google.colab import userdata, drive
import json

In [3]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Retrieve the token from Colab's Secrets Manager
hf_token = userdata.get('HF_TOKEN')

# Log in to HF using the retrieved token
login(hf_token)

In [4]:
# drive.mount('/content/drive')

# Working with Hugging Face Datasets

### Downloading files using HF datasets

In [ ]:
# Dwnloading the data from hugging face datasets.
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Clothing_Shoes_and_Jewelry", trust_remote_code=True)
items = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Clothing_Shoes_and_Jewelry", split="full", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
print(reviews["full"])
print(items["full"])

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 66033346
})

### Counting the Non-Null values in each column.

In [ ]:
def count_non_null(batch, columns):
    """Count non-null values in specified columns for a batch"""
    counts = {col: [] for col in columns}
    batch_size = len(batch[list(batch.keys())[0]]) # Get the size of the batch

    for i in range(batch_size):
        for col in columns:
            if col in batch:
                # Check if the value for the current example and column is not None
                # IN CASE OF PRICE COLUMN IT IS 'None' STRING and in case of features and descriptions it is '[]' empty list,
                # so do the appropriate changes accordingly in below code
                counts[col].append(1 if batch[col][i] is not None else 0)
            else:
                # If column not in batch, append 0 for this example
                counts[col].append(0)
    return counts

# Columns to check
columns = ['average_rating', 'rating_number']

# Count non-null values with multiprocessing
results = items.map(
    count_non_null,
    fn_kwargs={'columns': columns},
    batched=True,
    batch_size=1000,
    num_proc=4,  # Use 4 processes
    remove_columns=items.column_names  # Remove original columns to save memory
)

# Aggregate results
final_counts = {col: sum(results[col]) for col in columns}

# Print results
print("Non-null value counts:")
for col, count in final_counts.items():
    print(f"- {col}: {count:,} ({(count/len(items))*100:.1f}%)")

Map (num_proc=4):   0%|          | 0/7218481 [00:00<?, ? examples/s]

Non-null value counts:
- average_rating: 7,218,481.0 (100.0%)
- rating_number: 7,218,481 (100.0%)


In [ ]:
# NOT RECOMMENDED: This is a very bad way to check null values as the RAM will spike a lot.
column_name = 'helpful_vote'
x =reviews['full'][column_name]
print(x[:10])
add_ = 0
for i in x:
  if i == 0:
    add_ += 1
print(add_)

### Benchmarking the performance of HF datasets and DuckDB

In [ ]:
# To benchmark the difference in execution time of HF datasets and DuckDB!!!!
NPROC = min(8, os.cpu_count() or 2)

t0 = time.perf_counter()
# Filter keeps only verified rows (runs in parallel on CPU)
verified_ds = reviews.filter(lambda x: bool(x["verified_purchase"]), num_proc=NPROC)
count_ds = verified_ds.num_rows['full']
t1 = time.perf_counter()

print(f"[datasets] verified count = {count_ds:,}  | time = {t1 - t0:.2f}s  | num_proc={NPROC}")

In [ ]:
# TO use DuckDB we need to first convert the required data to parquet format for maximum efficiency.
parquet_dir = "/content/reviews_parquet"

# Keep only what we need for this quick benchmark to keep files tiny
need_cols = [c for c in reviews['full'].column_names if c in ("verified_purchase",)]
reviews_small = reviews['full'].remove_columns([c for c in reviews['full'].column_names if c not in need_cols])

# Write shards to Parquet (this creates multiple files under the folder)
reviews_small.to_parquet(parquet_dir)

In [ ]:
parquet_path = "/content/reviews_parquet"  # <-- use YOUR actual file path

con = duckdb.connect()
con.execute(f"PRAGMA threads={min(8, os.cpu_count() or 2)};")
con.execute("PRAGMA memory_limit='8GB';")

t0 = time.perf_counter()
count_duck = con.execute("""
  SELECT COUNT(*)
  FROM read_parquet(?)
  WHERE verified_purchase = TRUE
""", [parquet_path]).fetchone()[0]
t1 = time.perf_counter()

print(f"[duckdb ] verified count = {count_duck:,} | time = {t1 - t0:.2f}s")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[duckdb ] verified count = 62,175,766 | time = 2.16s


***CLEARLY DUCKDB IS BLAZINGLY FAST !!!***

### Preprocessing before converting to parquet.

In [ ]:
# REVIEWS — keep slim columns, filter verified
rev_keep = ["user_id","parent_asin","timestamp","rating","verified_purchase","helpful_vote"]
reviews_slim = reviews["full"].remove_columns([c for c in reviews["full"].column_names if c not in rev_keep])
reviews_slim = reviews_slim.filter(lambda x: bool(x["verified_purchase"]), num_proc=4)
# (Optional) drop the flag now that you’ve filtered:
reviews_slim = reviews_slim.remove_columns(["verified_purchase"])

Filter (num_proc=4):   0%|          | 0/66033346 [00:00<?, ? examples/s]

In [ ]:
# ITEMS — extract only what you need; we’ll map images→main_image_url later
itm_keep = ["parent_asin","main_category","title","average_rating","rating_number","price","images","categories","features","description","categories","details"]
items_slim = items.remove_columns([c for c in items.column_names if c not in itm_keep])

### Extracting the Main Image URL

In [ ]:
NPROC = min(8, os.cpu_count() or 2)

def _first_url(val):
    """Return the first non-empty string URL found inside val (str/list/ndarray/dict)."""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s or None
    if isinstance(val, (list, tuple, np.ndarray)):
        for x in val:
            u = _first_url(x)
            if u:
                return u
        return None
    if isinstance(val, dict):
        # common keys in the Amazon dumps; prioritize hi_res -> large -> medium -> url
        for k in ("hi_res", "large", "thumb"):
            if k in val:
                u = _first_url(val[k])
                if u:
                    return u
        return None
    # anything else
    return None

def to_main_url(batch):
    """datasets.map(batched=True) callback: reads batch['images'] (dicts) -> main_image_url"""
    out = []
    for img in batch["images"]:
        url = None
        if isinstance(img, dict):
            url = _first_url(img)           # dict case (your schema)
        else:
            url = _first_url(img)           # be tolerant to accidental list/str formats
        out.append(url)
    batch["main_image_url"] = out
    return batch


In [ ]:
# items_slim must contain the 'images' column
items_slim = items_slim.map(to_main_url, batched=True, num_proc=NPROC)
items_slim = items_slim.remove_columns(["images"])

In [ ]:
# Checking how many products there are without ant image.
len(items_slim.filter(lambda x: x["main_image_url"] is None, num_proc=NPROC))

### Converting to parquet format and saving locally and in google drive

In [ ]:
SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_path = f"{SAVE_DIR}/reviews_small_unpart"
itm_path = f"{SAVE_DIR}/items_small_unpart"

reviews_slim.to_parquet(rev_path)

size_bytes = os.path.getsize(rev_path)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
print("size:", human(size_bytes))

items_slim.to_parquet(itm_path)

Creating parquet from Arrow format:   0%|          | 0/62176 [00:00<?, ?ba/s]

4352307484

In [ ]:
# SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_HF_datasets'
# os.makedirs(SAVE_DIR, exist_ok=True)

# rev_g = f'{SAVE_DIR}/reviews_small_unpart'
# itm_g = f'{SAVE_DIR}/items_small_unpart'

# # write with a proper extension + compression
# reviews_slim.to_parquet(rev_g, engine='pyarrow', compression='snappy', index=False)
# items_slim.to_parquet(itm_g, engine='pyarrow', compression='snappy', index=False)

# # quick check
# import os
# print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
# print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

# Downloading the parquet file from Google Drive Link

In [ ]:
# Your shared links (file IDs extracted below)
REV_ID = "1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn"
ITM_ID = "155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq"

SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_OUT = f'{SAVE_DIR}/reviews_small_unpart'
ITM_OUT = f'{SAVE_DIR}/items_small_unpart'

gdown.download(id=REV_ID, output=REV_OUT, quiet=False)
gdown.download(id=ITM_ID, output=ITM_OUT, quiet=False)

# (optional) show sizes
def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print("reviews size:", human(os.path.getsize(REV_OUT)))
print("items   size:", human(os.path.getsize(ITM_OUT)))

Downloading...
From (original): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn
From (redirected): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn&confirm=t&uuid=f1dcb80b-1a8d-4b99-89ea-1d047a635e41
To: /content/artifacts/Post_HF_datasets/reviews_small_unpart
100%|██████████| 1.91G/1.91G [00:24<00:00, 78.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq
From (redirected): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq&confirm=t&uuid=3b9fdca4-bcc6-45d7-b5c1-038dbba32d51
To: /content/artifacts/Post_HF_datasets/items_small_unpart
100%|██████████| 4.20G/4.20G [00:46<00:00, 91.3MB/s]

reviews size: 1.78 GB
items   size: 3.91 GB


In [ ]:
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=REV_OUT)
items = load_dataset("parquet", data_files=ITM_OUT)

et = time.time()
print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 108.02244067192078 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Loading and Uploading data using HF

In [ ]:
api = HfApi()

repo_id = "PirateKing0402/Amazon_dataset"   # change this
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset')

### Uploading data to HF

In [ ]:
api.upload_file(
    path_or_fileobj=REV_OUT,
    path_in_repo="reviews_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add reviews parquet unpartitioned (hf_transfer)"
)

api.upload_file(
    path_or_fileobj=ITM_OUT,
    path_in_repo="items_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add items parquet unpartitioned (hf_transfer)"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/reviews_small_unpart         :   0%|          |  544kB / 1.91GB            

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/items_small_unpart           :   0%|          |  525kB / 4.20GB            

CommitInfo(commit_url='https://huggingface.co/datasets/PirateKing0402/Amazon_dataset/commit/a60281c9ad533dffaf04fc91a0b72f7ec7c29480', commit_message='Add items parquet unpartitioned (hf_transfer)', commit_description='', oid='a60281c9ad533dffaf04fc91a0b72f7ec7c29480', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset'), pr_revision=None, pr_num=None)

### Loading data from HF
( Slower than Google Drive download, better to use DuckDB to get data from HF )

In [ ]:
# Remove the reviews and items variables from memory
del reviews
del items

# You can optionally add print statements to confirm they are deleted (will raise NameError if successful)
# print(reviews)
# print(items)

In [ ]:
from datasets import load_dataset

# Define the repository ID and file paths within the repo
repo_id = "PirateKing0402/Amazon_dataset"
reviews_file_path = "reviews_small_unpart"
items_file_path = "items_small_unpart"
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{reviews_file_path}")
items = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{items_file_path}")
et = time.time()

print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

reviews_small_unpart:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

items_small_unpart:   0%|          | 0.00/4.20G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 969.474189043045 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Processing using DuckDB

### Using DuckDB on a data accessed through Remote Connection.
DuckDB's power lies in its ability to query massive remote datasets efficiently without local downloads. It achieves this by making many small, precise HTTP range requests to read only the parts of a file it needs.

This method is perfectly suited for cloud object stores like Amazon S3 or GCS, which are designed for this high-throughput access pattern. However, it triggers the anti-bot rate limits on standard web servers like the Hugging Face Hub, causing the query to fail.

In [ ]:
# ---- config: your repo + paths (file OR folder)
REPO_ID = "PirateKing0402/Amazon_dataset"
REV_PREFIX = "reviews_small_unpart"  # e.g. "reviews_small_unpart.parquet" OR a folder "reviews_small_unpart/"
ITM_PREFIX = "items_small_unpart"    # same idea for items

# Helper: build HTTPS URLs to the exact parquet files in the repo
def hf_parquet_urls(repo_id: str, prefix: str):
    files = list_repo_files(repo_id, repo_type="dataset")
    # case 1: a single file like "<prefix>.parquet"
    exact = [p for p in files if p == f"{prefix}"]
    if exact:
        return [f"https://huggingface.co/datasets/{repo_id}/resolve/main/{exact[0]}"]

rev_urls = hf_parquet_urls(REPO_ID, REV_PREFIX)
itm_urls = hf_parquet_urls(REPO_ID, ITM_PREFIX)

# Safety check: make sure we found files
print("review files:", len(rev_urls))
print("item files  :", len(itm_urls))
assert rev_urls, "No review parquet found in the repo/path you provided."
assert itm_urls, "No item parquet found in the repo/path you provided."

review files: 1
item files  : 1


In [ ]:
# ---- DuckDB setup (HTTP range reads; only needed bytes fetched)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='8GB';")  # optional

# ---- Items: count duplicate rows by parent_asin
sql_items = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup = con.execute(sql_items, {"urls": itm_urls}).fetchdf()

In [ ]:
sql_reviews = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup = con.execute(sql_reviews, {"urls": rev_urls}).fetchdf()

print("\nItems duplicates (pandas equivalence): duplicated(subset=['parent_asin']).sum()")
print(int(items_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup.loc[0, "keys_with_duplicates"]))

print("\nReviews duplicates (pandas equivalence): duplicated(subset=['user_id','parent_asin']).sum()")
print(int(reviews_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup.loc[0, "keys_with_duplicates"]))

### Using DuckDB on locally downloaded data.

In [ ]:
# ---- DuckDB setup (using local files)
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

# Define local file paths
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

In [ ]:
# ---- Items: count duplicate rows by parent_asin using local file
sql_items_local = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup_local = con.execute(sql_items_local, [ITM_PATH_LOCAL]).fetchdf()

print("Items duplicates (local file):")
print(int(items_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup_local.loc[0, "keys_with_duplicates"]))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items duplicates (local file):
0 | keys_with_duplicates: 0


In [ ]:
# ---- Reviews: count duplicate rows by user_id and parent_asin using local file
sql_reviews_local = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup_local = con.execute(sql_reviews_local, [REV_PATH_LOCAL]).fetchdf()

print("\nReviews duplicates (local file):")
print(int(reviews_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup_local.loc[0, "keys_with_duplicates"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Reviews duplicates (local file):
803670 | keys_with_duplicates: 684012


In [ ]:
# Using DuckDB on locally downloaded data to count rows with missing timestamps
sql_reviews_null_timestamp = """
WITH t AS (
  SELECT TRY_CAST("timestamp" AS BIGINT) AS ts
  FROM read_parquet(?)
)
SELECT COUNT(*) FROM t WHERE ts IS NULL;
"""
reviews_null_timestamp_count = con.execute(sql_reviews_null_timestamp, [REV_PATH_LOCAL]).fetchone()[0]

print(f"Number of rows with null timestamp in reviews (local file): {reviews_null_timestamp_count}")

Number of rows with null timestamp in reviews (local file): 0


In [ ]:
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

SAVE_DIR = '/content/artifacts/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_DEDUP_LOCAL = f"{SAVE_DIR}/reviews_small_unpart"          # <-- output file to create
ITM_DEDUP_LOCAL = f"{SAVE_DIR}/items_small_unpart"

os.makedirs(os.path.dirname(REV_DEDUP_LOCAL), exist_ok=True)
os.makedirs(os.path.dirname(ITM_DEDUP_LOCAL), exist_ok=True)

# 0) how many rows before
orig_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_PATH_LOCAL]).fetchone()[0]
print("rows before:", orig_cnt)

rows before: 62175766


In [ ]:
# 1) write a deduplicated copy:
#    - partition by (user_id, parent_asin)
#    - order by timestamp DESC so we keep the newest
#    - tie-breakers: helpful_vote DESC, rating DESC (optional but sensible)
#    - keep exactly one row: rn = 1
def sql_quote(path: str) -> str:
    # escape any single quotes in a file path for SQL
    return path.replace("'", "''")

src = sql_quote(REV_PATH_LOCAL)
dst = sql_quote(REV_DEDUP_LOCAL)

sql = f"""
COPY (
  WITH raw AS (
    SELECT
      user_id,
      parent_asin,
      TRY_CAST("timestamp" AS BIGINT) AS ts,  -- keep safe quoting
      rating,
      helpful_vote
    FROM read_parquet('{src}')
  ),
  ranked AS (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY user_id, parent_asin
        ORDER BY ts DESC NULLS LAST, helpful_vote DESC NULLS LAST, rating DESC NULLS LAST
      ) AS rn
    FROM raw
  )
  SELECT user_id, parent_asin, ts AS "timestamp", rating, helpful_vote
  FROM ranked
  WHERE rn = 1
) TO '{dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql)

# 2) check after
dedup_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_DEDUP_LOCAL]).fetchone()[0]
print("rows after :", dedup_cnt)
print("removed    :", orig_cnt - dedup_cnt)

# Copy the items data from the local path to the destination path
# Although we did not perform deduplication on items data,
# we are copying it to the Post_dedup folder for consistency with the reviews data.
itm_src = sql_quote(ITM_PATH_LOCAL)
itm_dst = sql_quote(ITM_DEDUP_LOCAL)

sql_copy_items = f"""
COPY (
  SELECT * FROM read_parquet('{itm_src}')
) TO '{itm_dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_copy_items)

print(f"Items data copied from {ITM_PATH_LOCAL} to {ITM_DEDUP_LOCAL}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows after : 61372096
removed    : 803670


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items data copied from /content/artifacts/Post_HF_datasets/items_small_unpart to /content/artifacts/Post_dedup/items_small_unpart


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g = f'{SAVE_DIR}/reviews_small_unpart'
itm_g = f'{SAVE_DIR}/items_small_unpart'

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_DEDUP_LOCAL}'))
TO '{rev_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{ITM_DEDUP_LOCAL}'))
TO '{itm_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# quick check
import os
print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reviews: 2541.530979156494 MB
items  : 3953.9629278182983 MB


In [ ]:
TimestampLike = Union[pd.Timestamp, str, int, float]

def temporal_split_ms_duckdb(
    src_parquet: str,
    time_col: str = "ts",
    *,
    test_fraction: Optional[float] = None,
    cutoff: Optional[TimestampLike] = None,
    train_includes_cutoff: bool = True,
    drop_na_time: bool = True,
    sort_within_splits: bool = False,
    out_prefix: Optional[str] = "splits/v1"
) -> Tuple[int, pd.Timestamp, int, int]:
    if (test_fraction is None) == (cutoff is None):
        raise ValueError("Provide exactly one of `test_fraction` or `cutoff`.")
    if test_fraction is not None:
        if not np.isfinite(test_fraction) or not (0.0 < float(test_fraction) < 1.0):
            raise ValueError("`test_fraction` must be finite and in (0,1).")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE TEMP VIEW _raw AS SELECT * FROM read_parquet('{src_parquet}')")

    where_clause = ""
    if drop_na_time:
        where_clause = f"WHERE TRY_CAST({time_col} AS BIGINT) IS NOT NULL"

    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW _clean AS
        SELECT *, TRY_CAST({time_col} AS BIGINT) AS ms
        FROM _raw
        {where_clause}
    """)
    mn, mx, n = con.execute("SELECT MIN(ms), MAX(ms), COUNT(*) FROM _clean").fetchone()
    if n == 0:
        raise ValueError("All timestamps are NaN after parsing; nothing to split.")

    if cutoff is None:
        q = 1.0 - float(test_fraction)
        cutoff_ms = int(con.execute("SELECT quantile_disc(ms, ?) FROM _clean", [q]).fetchone()[0])
    else:
        if isinstance(cutoff, (int, float)) and np.isfinite(cutoff):
            cutoff_ms = int(cutoff)
        else:
            cutoff_ms = int(pd.to_datetime(cutoff, utc=True).value // 1_000_000)

    if not (mn <= cutoff_ms <= mx):
        raise RuntimeError(f"Cutoff {cutoff_ms} outside data range [{mn}, {mx}].")

    op_train = "<=" if train_includes_cutoff else "<"
    op_test  = ">"  if train_includes_cutoff else ">="

    con.execute(f"CREATE OR REPLACE TEMP VIEW train AS SELECT * FROM _clean WHERE ms {op_train} {cutoff_ms}")
    con.execute(f"CREATE OR REPLACE TEMP VIEW test  AS SELECT * FROM _clean WHERE ms {op_test}  {cutoff_ms}")

    train_n = con.execute("SELECT COUNT(*) FROM train").fetchone()[0]
    test_n  = con.execute("SELECT COUNT(*) FROM test").fetchone()[0]
    if train_n == 0 or test_n == 0:
        raise RuntimeError(f"Empty split: train={train_n}, test={test_n}. Adjust `test_fraction`/`cutoff`.")

    if out_prefix:
        train_query = "SELECT * FROM train ORDER BY ms" if sort_within_splits else "SELECT * FROM train"
        test_query  = "SELECT * FROM test  ORDER BY ms" if sort_within_splits else "SELECT * FROM test"

        con.execute(f"COPY ({train_query}) TO '{out_prefix}/reviews_small_train' (FORMAT PARQUET)")
        con.execute(f"COPY ({test_query})  TO '{out_prefix}/reviews_small_test'  (FORMAT PARQUET)")


    cutoff_ts = pd.to_datetime(cutoff_ms, unit="ms", utc=True).tz_convert(None)
    return cutoff_ms, cutoff_ts, train_n, test_n

In [ ]:
# Create the output directory if it doesn't exist
output_dir = '/content/artifacts/Post_temporal_splits'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

cutoff_ms, cutoff_ts, n_tr, n_te = temporal_split_ms_duckdb(
    '/content/artifacts/Post_dedup/reviews_small_unpart',
    time_col='timestamp',
    # test_fraction=0.20,
    cutoff = "01-01-2022",
    train_includes_cutoff=True,
    sort_within_splits=True,
    out_prefix=output_dir
)

print(f"Temporal split complete. Cutoff date: {cutoff_ts}")
print(f"Train split rows: {n_tr:,}")
print(f"Test split rows: {n_te:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporal split complete. Cutoff date: 2022-01-01 00:00:00
Train split rows: 49,629,909
Test split rows: 11,742,187


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_temporal_splits'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train'
rev_g_test = f'{SAVE_DIR}/reviews_small_test'

REV_SPLIT_TRAIN = "/content/artifacts/Post_temporal_splits/reviews_small_train"
REV_SPLIT_TEST = "/content/artifacts/Post_temporal_splits/reviews_small_test"

sql_copy_reviews_train = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_train)

sql_copy_reviews_test = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TEST}'))
TO '{rev_g_test}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_test)

# quick check
import os
print('reviews_train:', os.path.getsize(rev_g_train)/1024**2, 'MB')
print('items_test  :', os.path.getsize(rev_g_test)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reviews_train: 2057.8552207946777 MB
items_test  : 489.0047264099121 MB


In [ ]:
def kcore_filter_iterative_duckdb(
    src_parquet: str,
    *,
    user_col: str = "user_id",
    item_col: str = "parent_asin",
    user_k: int = 5,
    item_k: int = 5,
    max_iters: int = 100,
    out_path: Optional[str] = "/content/splits/kcore_train",
    return_history: bool = True,
    return_df: bool = False,
    verbose: bool = False,
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    def q(ident: str) -> str:
        if '"' in ident:
            raise ValueError(f'Identifier {ident!r} contains a double quote (").')
        return f'"{ident}"'
    u, i = q(user_col), q(item_col)

    con = duckdb.connect()
    con.execute("PRAGMA disable_progress_bar")
    src = src_parquet.rstrip('/')

    # Start with a TABLE (materialized), not a view
    con.execute(f"CREATE OR REPLACE TEMP TABLE cur AS SELECT * FROM read_parquet('{src}')")
    n0 = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]
    if n0 == 0:
        raise ValueError("No rows to process.")

    history: List[Dict] = []

    for it in range(1, max_iters + 1):
        if verbose: print(f"Iteration {it} started")
        n_before = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]

        # User prune
        con.execute("DROP TABLE IF EXISTS ucnt")
        con.execute(f"CREATE TEMP TABLE ucnt AS SELECT {u} AS u, COUNT(*) AS c FROM cur GROUP BY {u}")
        con.execute("DROP TABLE IF EXISTS cur_u")
        con.execute(f"""
            CREATE TEMP TABLE cur_u AS
            SELECT c.* FROM cur c
            JOIN ucnt u ON c.{user_col} = u.u
            WHERE u.c >= {user_k}
        """)
        n_u = con.execute("SELECT COUNT(*) FROM cur_u").fetchone()[0]
        if n_u == 0:
            raise RuntimeError(f"All rows pruned at user step (iter={it}). Lower user_k/item_k.")

        # Item prune
        con.execute("DROP TABLE IF EXISTS icnt")
        con.execute(f"CREATE TEMP TABLE icnt AS SELECT {i} AS v, COUNT(*) AS c FROM cur_u GROUP BY {i}")
        con.execute("DROP TABLE IF EXISTS nxt")
        con.execute(f"""
            CREATE TEMP TABLE nxt AS
            SELECT u.* FROM cur_u u
            JOIN icnt v ON u.{item_col} = v.v
            WHERE v.c >= {item_k}
        """)
        n_after = con.execute("SELECT COUNT(*) FROM nxt").fetchone()[0]
        if n_after == 0:
            raise RuntimeError(f"All rows pruned at item step (iter={it}). Lower user_k/item_k.")

        users_after, items_after = con.execute(f"""
            SELECT COUNT(DISTINCT {u}), COUNT(DISTINCT {i}) FROM nxt
        """).fetchone()

        history.append({
            "iter": it,
            "rows_before": n_before,
            "rows_after": n_after,
            "users_after": users_after,
            "items_after": items_after,
            "removed": n_before - n_after,
        })

        # Convergence: no change this iteration
        if n_after == n_before:
            con.execute("DROP TABLE IF EXISTS cur")
            con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")
            if verbose: print(f"Iteration {it} finished (converged).")
            break

        # Prepare next iteration: replace cur with nxt (tables, so no cycles)
        con.execute("DROP TABLE IF EXISTS cur")
        con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")

    # Persist and/or return
    if out_path:
        con.execute(f"COPY (SELECT * FROM cur) TO '{out_path}' (FORMAT PARQUET)")
    out_df = con.execute("SELECT * FROM cur").df() if return_df else None
    hist_df = pd.DataFrame(history) if return_history else None
    return out_df, hist_df

In [ ]:
SAVE_DIR = '/content/artifacts/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)
user_k = 6
item_k = 6
out_path = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

filtered_df, history = kcore_filter_iterative_duckdb(
    src_parquet='/content/artifacts/Post_temporal_splits/reviews_small_train',  # or a folder with *.parquet
    user_col='user_id',
    item_col='parent_asin',
    user_k=user_k,
    item_k=item_k,
    max_iters=10,            # your data is already deduped earlier
    out_path=out_path,
    return_history=True,
    verbose=True,
    return_df=False                   # avoid pulling huge data back to RAM
)

Iteration 1 started
Iteration 2 started
Iteration 3 started
Iteration 4 started
Iteration 5 started
Iteration 6 started
Iteration 7 started
Iteration 8 started
Iteration 9 started
Iteration 9 finished (converged).


In [ ]:
history

,iter,rows_before,rows_after,users_after,items_after,removed
0,1,49629909,14213565,1817343,412915,35416344
1,2,14213565,11185349,1171160,336006,3028216
2,3,11185349,10874091,1115865,327925,311258
3,4,10874091,10836376,1109271,326959,37715
4,5,10836376,10831756,1108476,326830,4620
5,6,10831756,10831121,1108365,326814,635
6,7,10831121,10831061,1108356,326811,60
7,8,10831061,10831051,1108354,326811,10
8,9,10831051,10831051,1108354,326811,0


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

REV_K_CORE_TRAIN = f"/content/artifacts/Post_k_core_filtering/reviews_small_train_{user_k}_{item_k}"

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_K_CORE_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

# quick check
import os
print('reviews_after_k_core_filtering:', os.path.getsize(rev_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reviews_after_k_core_filtering: 2541.530979156494 MB


# Content Based Recommender system

In [5]:
# Define the file ID from the shared link
file_id1 = "1M9wKeGUTAQ9ZFpwDd4AkhbdS0NfdA-0U"
file_id2 = "1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p"

# Define the local path to save the downloaded file
SAVE_DIR = '/content/artifacts/Working'
os.makedirs(SAVE_DIR, exist_ok=True)
local_file_path1 = f'{SAVE_DIR}/reviews_small_train' # You can rename the file as needed
local_file_path2 = f'{SAVE_DIR}/items_small'

# Download the file
print(f"Downloading file with ID: {file_id1} to {local_file_path1}")
gdown.download(id=file_id1, output=local_file_path1, quiet=False)

# Dont load this unnecessarily takes up RAM, because we will use DuckDB and it works acess file directly from hard_disk from the filepath, so just install the file at the file path.
# # Load the downloaded parquet file into a dataset
# print(f"Loading dataset from {local_file_path1}")
# reviews_train_dataset = load_dataset("parquet", data_files=local_file_path1)

# # Display information about the loaded dataset
# print("reviews train dataset:")
# print(reviews_train_dataset)

# Download the file
print(f"Downloading file with ID: {file_id2} to {local_file_path2}")
gdown.download(id=file_id2, output=local_file_path2, quiet=False)

# # Load the downloaded parquet file into a dataset
# print(f"Loading dataset from {local_file_path2}")
# items_dataset = load_dataset("parquet", data_files=local_file_path2)

# # Display information about the loaded dataset
# print("Items dataset:")
# print(items_dataset)


Downloading...
From (original): https://drive.google.com/uc?id=1M9wKeGUTAQ9ZFpwDd4AkhbdS0NfdA-0U
From (redirected): https://drive.google.com/uc?id=1M9wKeGUTAQ9ZFpwDd4AkhbdS0NfdA-0U&confirm=t&uuid=bf51033e-9a78-4d92-bc6f-3c0620646462
To: /content/artifacts/Working/reviews_small_train
100%|██████████| 2.16G/2.16G [00:22<00:00, 95.2MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p
From (redirected): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p&confirm=t&uuid=454525bc-5465-4886-8594-9b10b3e009f4
To: /content/artifacts/Working/items_small
100%|██████████| 4.15G/4.15G [00:43<00:00, 95.9MB/s]


'/content/artifacts/Working/items_small'

In [6]:
# Establish DuckDB connection
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

In [ ]:
# Execute the query and fetch all results into a Python list
initial_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{local_file_path2}')").fetchone()[0]
print(initial_count)

7218481


In [7]:
# Filter items data to remove rows where main_image_url is null using DuckDB

ITM_FILTERED_LOCAL = f"{SAVE_DIR}/items_small_filtered"

# SQL query to filter and copy the data
sql_filter_items = f"""
COPY (
  SELECT *
  FROM read_parquet('{local_file_path2}')
  WHERE main_image_url IS NOT NULL
) TO '{ITM_FILTERED_LOCAL}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_filter_items)

# Quick check of the number of rows after filtering
filtered_items_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ITM_FILTERED_LOCAL}')").fetchone()[0]
print(f"Number of rows in items data after filtering for main_image_url: {filtered_items_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of rows in items data after filtering for main_image_url: 7208433


In [ ]:
temp4 = con.execute(f"SELECT main_image_url from read_parquet('{ITM_FILTERED_LOCAL}')").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
for item in temp4:
  # The URL is the first element of the tuple
  url = item[0] if isinstance(item, tuple) and len(item) > 0 else "no no no"
  if url and isinstance(url, str) and url.startswith('http'):
    pass
  else:
    print(f"Found a URL that does not start with 'http' or is not a valid URL: {url}")
    print(type(url))

print("Finished checking URLs.")

Finished checking URLs.


In [ ]:
# # Define local file paths for the filtered items and the training reviews
# ITM_FILTERED_LOCAL = f"{SAVE_DIR}/items_small_filtered"
# ITM_REVIEW_FILTERED_LOCAL = f"{SAVE_DIR}/items_review_filtered"

# # SQL query to filter items based on the existence of their parent_asin in the reviews_train data
# sql_filter_items_by_reviews = f"""
# COPY (
#   SELECT DISTINCT i.*
#   FROM read_parquet('{ITM_FILTERED_LOCAL}') AS i
#   JOIN read_parquet('{local_file_path1}') AS r
#   ON i.parent_asin = r.parent_asin
# ) TO '{ITM_REVIEW_FILTERED_LOCAL}'
#   (FORMAT PARQUET, COMPRESSION ZSTD);
# """

# print("Filtering items based on reviews in training data...")
# con.execute(sql_filter_items_by_reviews)
# print(f"Filtered items saved to: {ITM_REVIEW_FILTERED_LOCAL}")

# # Optional: Quick check of the number of rows after filtering
# filtered_items_by_reviews_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ITM_REVIEW_FILTERED_LOCAL}')").fetchone()[0]
# print(f"Number of rows in items data after filtering by reviews: {filtered_items_by_reviews_count}")

Filtering items based on reviews in training data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtered items saved to: /content/artifacts/Working/items_review_filtered
Number of rows in items data after filtering by reviews: 6075758


In [8]:
temp3 = con.execute(f"SELECT categories,main_image_url,details from read_parquet('{ITM_FILTERED_LOCAL}')").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [15]:
temp3[:3]

[(['Clothing, Shoes & Jewelry',
   'Women',
   'Clothing',
   'Swimsuits & Cover Ups',
   'Cover-Ups'],
  'https://m.media-amazon.com/images/I/61KIZjb54AL._AC_UL1500_.jpg',
  '{"Department": "womens", "Date First Available": "April 3, 2022"}'),
 (['Clothing, Shoes & Jewelry',
   'Women',
   'Shoes',
   'Outdoor',
   'Hiking & Trekking',
   'Hiking Shoes'],
  'https://m.media-amazon.com/images/I/81oltcdwYSL._AC_UL1500_.jpg',
  '{"Package Dimensions": "14.02 x 9.29 x 4.8 inches; 1 Pounds", "Item model number": "00-3C4Q7W8I-AX", "Department": "womens", "Date First Available": "November 2, 2017"}'),
 (['Clothing, Shoes & Jewelry', 'Women', 'Shoes', 'Sandals', 'Flats'],
  'https://m.media-amazon.com/images/I/61Q8PVmaltL._AC_UL1500_.jpg',
  '{"Product Dimensions": "10 x 15 x 6 inches; 2 Pounds", "Item model number": "1570-891", "Department": "womens", "Date First Available": "March 22, 2017", "Manufacturer": "SAS"}')]

In [18]:
t = {}
for i in temp3:
  element = json.loads(i[2])
  for key in element.keys():
    if "weight" in key.lower():
      if key in t:
        t[key] += 1
      else:
        t[key] = 1

In [19]:
t

{'Item Weight': 454849,
 'Package Weight': 76848,
 'Weight Limit': 446,
 'Stone Weight': 329,
 'Weight': 541,
 'Total Diamond Weight': 325,
 'Minimum weight recommendation': 270,
 'Maximum weight recommendation': 285,
 'Maximum Weight Recommendation': 468,
 'Minimum Weight Recommendation': 13,
 'Total gem weight': 80,
 'Item Weight 5.7 Ounces': 1,
 'Maximum Weight Capacity': 9,
 'Battery Weight': 5,
 'Total Metal Weight': 4,
 'Line Weight': 7,
 'Item Weight 30 Pounds': 1,
 'Item Weight 0.71 Pounds': 1,
 'Item Weight 13 Pounds': 2,
 'Paper Weight': 4,
 'Item Weight 1 Pounds': 2,
 'Item Weight 1.92 Pounds': 1,
 'Item Weight 16 Grams': 1,
 'Net Content Weight': 1,
 'Item Weight 7.5 Pounds': 1,
 'Item Weight 1 Grams': 1,
 'Item Weight 0.05 Pounds': 1,
 'Item Weight 1 Ounces': 1,
 'Minimum Weight Capacity': 5,
 'Item Weight 10.4 Ounces': 1,
 'Item Weight 0.19 Kilograms': 1,
 'Maximum Weight Recommendation 3 Pounds': 1,
 'Item Weight 3 Pounds': 1,
 'Item Weight 0.55 Pounds': 1,
 'Maximum Wei

In [ ]:
7208433 - 909679

6298754

In [ ]:
counter = 0
p = {}
for i in temp3:
  element = json.loads(i[2])
  if "Department" in element:
    if element["Department"].lower() in p:
      p[element["Department"].lower()] += 1
    else:
      p[element["Department"].lower()] = 1
  else:
    counter += 1

counter

909679

In [ ]:
h = dict(sorted(p.items(), key=lambda item: item[1], reverse=True))

In [ ]:
count = 0
ans = 0
for val in h.values():
  ans += val
  count += 1
  if count == 13:
    break

In [ ]:
ans

6254124

In [10]:
ITM_FILTERED_LOCAL

'/content/artifacts/Working/items_small_filtered'

In [11]:
items_with_gender_clean = "/content/artifacts/Working/items_with_gender_clean"

os.makedirs(os.path.dirname(items_with_gender_clean), exist_ok=True)


sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{ITM_FILTERED_LOCAL}')
  ),
  x AS (
    SELECT
      *,
      regexp_extract(CAST(details AS VARCHAR),
                     '(?i)"department"\s*:\s*"([^"]+)"', 1) AS gender_raw
    FROM src
    WHERE regexp_matches(CAST(details AS VARCHAR),
                         '(?i)"department"\s*:\s*"')
  ),
  norm AS (
    SELECT
      *,
      lower(
        regexp_replace(
          regexp_replace(gender_raw, '[^A-Za-z -]+', ''),   -- keep letters / space / hyphen
          '\s+', ' '                                       -- collapse whitespace
        )
      ) AS dep_norm
    FROM x
  ),
  map AS (
    SELECT
      *,
      CASE
        -- specific buckets first
        WHEN dep_norm LIKE '%baby%girl%' OR dep_norm LIKE '%girl%baby%' THEN 'baby-girls'
        WHEN dep_norm LIKE '%baby%boy%'  OR dep_norm LIKE '%boy%baby%'  THEN 'baby-boys'
        WHEN dep_norm LIKE '%unisex%baby%'                               THEN 'unisex-baby'
        WHEN dep_norm LIKE '%unisex%child%'                              THEN 'unisex-child'
        WHEN dep_norm LIKE '%unisex%adult%'                              THEN 'unisex-adult'
        -- canonicalize women/men to womens/mens
        WHEN dep_norm LIKE '%womens%' OR dep_norm LIKE '%women%'         THEN 'womens'
        WHEN dep_norm LIKE '%mens%'   OR dep_norm LIKE '%men%'           THEN 'mens'
        WHEN dep_norm LIKE '%girls%'  OR dep_norm LIKE '%girl%'          THEN 'girls'
        WHEN dep_norm LIKE '%boys%'   OR dep_norm LIKE '%boy%'           THEN 'boys'
        ELSE NULL
      END AS gender
    FROM norm
  ),
  kept AS (
    SELECT *
    FROM map
    WHERE gender IN (
      'womens','mens','girls','boys',
      'baby-girls','baby-boys',
      'unisex-adult','unisex-child','unisex-baby'
    )
  )
  SELECT * EXCLUDE (dep_norm, gender_raw) FROM kept
) TO '{items_with_gender_clean}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_gender_clean)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_gender_clean


In [12]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_gender_clean}')").fetchone()[0]

6271634

In [20]:
OUT_PARQUET = "/content/artifacts/Working/items_with_dimension_weight"
os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")

# 0) Source view with text-cast once
con.execute(f"""
CREATE OR REPLACE TEMP VIEW src AS
SELECT *, CAST(details AS VARCHAR) AS dtxt
FROM read_parquet('{items_with_gender_clean}');
""")
print("src count:", con.execute("SELECT COUNT(*) FROM src").fetchone()[0])

src count: 6271634


In [46]:
# 1) dims: extract raw dimension & weight strings (no normalization)
sql_dims = r"""
CREATE OR REPLACE TEMP VIEW dims AS
SELECT
  *,
  -- first non-null dimension string (keep raw text)
  COALESCE(
    regexp_extract(dtxt, '(?is)"package\s+dimensions"[^:]*:\s*"([^"]+)"', 1),
    regexp_extract(dtxt, '(?is)"product\s+dimensions"[^:]*:\s*"([^"]+)"', 1),
    regexp_extract(dtxt, '(?is)"item\s+dimensions\s*lxwxh"[^:]*:\s*"([^"]+)"', 1),
    regexp_extract(dtxt, '(?is)"item\s*package\s+dimensions\s*l\s*x\s*w\s*x\s*h"[^:]*:\s*"([^"]+)"', 1),
  ) AS dimensions_raw,
  -- prefer item weight; fallback to package weight
  COALESCE(
    regexp_extract(dtxt, '(?i)"item weight"\s*:\s*"([^"]+)"', 1),
    regexp_extract(dtxt, '(?i)"package weight"\s*:\s*"([^"]+)"', 1)
  ) AS weight_raw
FROM src;
"""
con.execute(sql_dims)

df = con.execute("""
SELECT * FROM dims LIMIt 100000;
""").fetchdf()

In [50]:
for i in range(10000):
  e = json.loads(df.iloc[i]["details"])
  for key in e.keys():
    if "Item Package" in key or "Item Dimensions LxWxH" in key:
      print([df.iloc[i]["details"],df.iloc[i]["dimensions_raw"],df.iloc[i]["weight_raw"]])

['{"Brand": "PEARL IZUMI", "Material": "Faux Leather, Silicone", "Size": "Large", "Sport": "Cycling", "Age Range (Description)": "Adult", "Color": "Black", "Special Feature": "Water Resistant, Thermal", "Hand Orientation": "Ambidextrous", "Item Package Quantity": "1", "Product Care Instructions": "Hand Wash", "Is Discontinued By Manufacturer": "Yes", "Package Dimensions": "1 x 1 x 1 inches; 1 Pounds", "Item model number": "PIMSShellG-P", "Department": "mens", "Date First Available": "June 16, 2012", "Manufacturer": "PEARL iZUMi"}', '1 x 1 x 1 inches; 1 Pounds', '']
['{"Brand": "Columbia", "Material": "Polyester", "Size": "X-Large", "Sport": "Snowshoeing", "Color": "Black", "Item Package Quantity": "1", "Product Care Instructions": "Machine Wash", "Package Dimensions": "13.9 x 5.3 x 2.7 inches; 6.4 Ounces", "Item model number": "1683921053", "Department": "Mens", "Date First Available": "November 30, 2015", "Manufacturer": "Columbia (Sporting Goods)"}', '13.9 x 5.3 x 2.7 inches; 6.4 Oun

In [ ]:
7208433 - 3786834

3421599

In [ ]:
counter = 0
for i in temp3:
  element = json.loads(i[2])
  if "Manufacturer" not in element:
    counter += 1

counter

3786834

In [ ]:
7208433 - 26257036

-19048603

In [ ]:
counter = 0
c = {}
for i in temp3:
  element = json.loads(i[2])
  for key in element.keys():
    if "dimension" in key.lower():
      if key in c:
        c[key] += 1
      else:
        c[key] = 1
      counter += 1
      break

counter

5484631

In [ ]:
3921530 + 1474981

5396511

In [ ]:
c

{'Package Dimensions': 3921530,
 'Product Dimensions': 1474981,
 'Item Package Dimensions L x W x H': 76242,
 'Item Dimensions LxWxH': 8780,
 'Item Dimensions  LxWxH': 590,
 'Dimensions': 2494,
 'Item Display Dimensions': 14}

In [ ]:
if "rahul" in "Rahul".lower():
  print("yes")

yes


In [ ]:
counter = 0
a = []
for i in temp3:
  # Check if any string in the list i[0] (categories) contains "Clothing", "Shoes", or "Jewelry" (case-insensitive)
  for element in i[0]:
    if "clothing, shoes & jewelry" not in element.lower() and "clothing" in element.lower():
      counter += 1
      a.append(element)
      break
    elif "clothing, shoes & jewelry" not in element.lower() and "shoes" in element.lower():
      a.append(element)
      counter += 1
      break
    elif "clothing, shoes & jewelry" not in element.lower() and "jewelry" in element.lower():
      a.append(element)
      counter += 1
      break

print(counter)

5628345


In [ ]:
b = {}
for i in a:
  if i in b:
    b[i] += 1
  else:
    b[i] = 1

In [ ]:
2957375 + 1689150 + 744781

5391306

In [ ]:
sorted_b_by_value = dict(sorted(b.items(), key=lambda item: item[1], reverse=True))
sorted_b_by_value

{'Clothing': 2957375,
 'Shoes': 1689150,
 'Jewelry': 744781,
 'Sport Specific Clothing': 164088,
 'Shoe, Jewelry & Watch Accessories': 59920,
 'Kids Shoes Union': 4978,
 'Designer Shoes & Handbags': 1115,
 "Kids' Clothing": 747,
 'Prime Day: 30% off Select Clothing, Shoes & More for Men, Women, Kids & Baby': 715,
 'Unisex Baby Clothing': 693,
 'Kids Shoes': 597,
 'Clothing Outlet': 448,
 "Women's Shoes": 287,
 "Women's Comfort Shoes FA Test": 223,
 'TurboTax Refund Bonus Exclusive: 20% Off Select Shoes': 159,
 'Plain Silver Jewelry': 157,
 'Free One-Day Shipping in Shoes & Handbags': 136,
 'Comfort Shoes': 135,
 'Pearl Jewelry': 116,
 'Amazon Fashion: Clothing': 102,
 'Up to 25% off Festive Jewelry': 102,
 'Up to 30% off Shoes, Handbags and More': 101,
 '20% Off $100 Shoes for Women': 95,
 'Diamond Jewelry Store': 93,
 'Jewelry Gifts': 77,
 'TEST W GENZ SHOES': 76,
 'Jewelry & Watches Outlet': 65,
 'TurboTax Refund Bonus Exclusive: 20% Off Select Clothing': 59,
 '20% Off Athletic & Out

In [ ]:
counter = 0
for i in temp3[:100]:
  print(i[0])


['Clothing, Shoes & Jewelry', 'Women', 'Clothing', 'Swimsuits & Cover Ups', 'Cover-Ups']
['Clothing, Shoes & Jewelry', 'Women', 'Shoes', 'Outdoor', 'Hiking & Trekking', 'Hiking Shoes']
['Clothing, Shoes & Jewelry', 'Women', 'Shoes', 'Sandals', 'Flats']
['Clothing, Shoes & Jewelry', 'Novelty & More', 'Clothing', 'Novelty', 'Women', 'Skirts']
['Clothing, Shoes & Jewelry', 'Women', 'Handbags & Wallets', 'Crossbody Bags']
['Clothing, Shoes & Jewelry', 'Women', 'Jewelry', 'Rings']
['Clothing, Shoes & Jewelry', 'Women', 'Shoes', 'Loafers & Slip-Ons']
['Clothing, Shoes & Jewelry', 'Novelty & More', 'Clothing', 'Novelty', 'Boys', 'Accessories', 'Gloves & Mittens']
['Clothing, Shoes & Jewelry', 'Women', 'Clothing', 'Dresses', 'Formal']
['Clothing, Shoes & Jewelry', 'Luggage & Travel Gear', 'Travel Accessories']
['Clothing, Shoes & Jewelry', 'Women', 'Clothing', 'Dresses', 'Casual']
['Clothing, Shoes & Jewelry', 'Women', 'Clothing', 'Active', 'Active Shorts']
['Clothing, Shoes & Jewelry', 'Men',